# U.S. Recession Forecasting — Feature Engineering

This notebook transforms the monthly macroeconomic data into features suitable for recession forecasting.

The main objective is to predict whether the U.S. economy will be in recession six months in the future using economic information available at the current point in time.

Rather than relying only on the current levels of economic indicators, the feature set also captures changes, growth rates and recent trends.


In [51]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [52]:
df = pd.read_csv(
    "../data/processed/macroeconomic_monthly.csv",
    index_col=0,
    parse_dates=True
)

df.head()


,unemployment_rate,initial_claims,cpi,fed_funds_rate,yield_spread_10y_2y,yield_spread_10y_3m,industrial_production,housing_starts,consumer_sentiment,credit_spread,recession
1986-01-01,6.7,366750.0,109.9,8.14,1.051429,1.891905,55.7481,1972.0,95.6,2.241429,0.0
1986-02-01,7.2,369500.0,109.7,7.86,0.736842,1.413684,55.3568,1848.0,95.9,2.408421,0.0
1986-03-01,7.2,381200.0,109.1,7.48,0.564000,1.014500,54.9625,1876.0,95.1,2.716500,0.0
1986-04-01,7.1,385750.0,108.7,6.99,0.604091,1.063636,55.0413,1933.0,96.2,2.891364,0.0
1986-05-01,7.2,386000.0,109.0,6.85,0.642381,1.374762,55.1226,1854.0,94.8,2.576190,0.0


In [53]:
df.shape


(487, 11)

## 1. Forecasting Target

The original recession variable describes whether the economy is currently in recession. For forecasting, the target must instead represent the state of the economy in the future.

The main forecasting horizon is six months. Therefore, for each month, the target indicates whether the economy will be in recession six months later.


In [54]:
forecast_horizon = 6

df["recession_6m"] = df["recession"].shift(-forecast_horizon)


In [55]:
df[["recession", "recession_6m"]].tail(12)


,recession,recession_6m
2025-08-01,0.0,0.0
2025-09-01,0.0,0.0
2025-11-01,0.0,0.0
2025-12-01,0.0,0.0
2026-01-01,0.0,0.0
2026-02-01,0.0,0.0
2026-03-01,0.0,NaN
2026-04-01,0.0,NaN
2026-05-01,0.0,NaN
2026-06-01,0.0,NaN


The target is shifted backwards by six rows so that the economic indicators observed in month *t* are paired with the recession status in month *t + 6*.


In [56]:
example = df.loc[
    "2019-07-01":"2020-06-01",
    ["recession", "recession_6m"]
]

example


,recession,recession_6m
2019-07-01,0.0,0.0
2019-08-01,0.0,0.0
2019-09-01,0.0,1.0
2019-10-01,0.0,1.0
2019-11-01,0.0,0.0
2019-12-01,0.0,0.0
2020-01-01,0.0,0.0
2020-02-01,0.0,0.0
2020-03-01,1.0,0.0
2020-04-01,1.0,0.0


The 2020 recession provides a useful sanity check. Since the recession begins in March 2020 in this dataset, September 2019 should have a six-month-ahead recession target equal to 1.


## 2. Feature Construction

Economic indicators can contain useful information both in their current levels and in how they have recently changed. I therefore construct features representing current conditions, changes and growth rates.


In [57]:
features = pd.DataFrame(index=df.index)


### Current Levels

Variables whose levels have a direct economic interpretation are retained in their original form.


In [58]:
level_features = [
    "unemployment_rate",
    "fed_funds_rate",
    "yield_spread_10y_2y",
    "yield_spread_10y_3m",
    "consumer_sentiment",
    "credit_spread"
]

for column in level_features:
    features[column] = df[column]

features.head()


,unemployment_rate,fed_funds_rate,yield_spread_10y_2y,yield_spread_10y_3m,consumer_sentiment,credit_spread
1986-01-01,6.7,8.14,1.051429,1.891905,95.6,2.241429
1986-02-01,7.2,7.86,0.736842,1.413684,95.9,2.408421
1986-03-01,7.2,7.48,0.564000,1.014500,95.1,2.716500
1986-04-01,7.1,6.99,0.604091,1.063636,96.2,2.891364
1986-05-01,7.2,6.85,0.642381,1.374762,94.8,2.576190


### Unemployment Changes

Changes in unemployment may capture deterioration or improvement in labor market conditions that is not visible from the unemployment level alone.


In [59]:
features["unemployment_change_3m"] = df["unemployment_rate"].diff(3)
features["unemployment_change_6m"] = df["unemployment_rate"].diff(6)
features["unemployment_change_12m"] = df["unemployment_rate"].diff(12)


### Initial Unemployment Claims

Initial unemployment claims provide relatively high-frequency information about labor market conditions. Percentage changes are used to capture whether claims are increasing or decreasing over time.


In [60]:
features["initial_claims"] = df["initial_claims"]

features["initial_claims_change_3m"] = (
    df["initial_claims"].pct_change(3) * 100
)

features["initial_claims_change_6m"] = (
    df["initial_claims"].pct_change(6) * 100
)


### Inflation

The CPI level generally trends upward over time, so its raw value is less useful for comparing economic conditions across decades. Year-over-year percentage change is used as an inflation measure instead.


In [61]:
features["inflation_yoy"] = (
    df["cpi"].pct_change(12) * 100
)


### Industrial Production

Growth in industrial production provides information about changes in economic activity. Both short-term and annual growth rates are included.


In [62]:
features["industrial_production_growth_3m"] = (
    df["industrial_production"].pct_change(3) * 100
)

features["industrial_production_growth_12m"] = (
    df["industrial_production"].pct_change(12) * 100
)


### Housing Activity

Housing starts are transformed into percentage changes to capture changes in residential construction activity.


In [63]:
features["housing_starts_change_3m"] = (
    df["housing_starts"].pct_change(3) * 100
)

features["housing_starts_change_12m"] = (
    df["housing_starts"].pct_change(12) * 100
)


### Consumer Sentiment Changes

In addition to the current level of consumer sentiment, changes over recent months are included to capture improving or deteriorating consumer expectations.


In [64]:
features["consumer_sentiment_change_3m"] = (
    df["consumer_sentiment"].diff(3)
)

features["consumer_sentiment_change_6m"] = (
    df["consumer_sentiment"].diff(6)
)


### Yield Curve Inversion

In addition to the continuous yield spreads, binary variables are created to indicate whether each yield spread is negative. A negative spread represents an inverted yield curve.


In [65]:
features["yield_curve_inverted_10y_2y"] = (
    df["yield_spread_10y_2y"] < 0
).astype(int)

features["yield_curve_inverted_10y_3m"] = (
    df["yield_spread_10y_3m"] < 0
).astype(int)


### Credit Spread Changes

Changes in the corporate credit spread are included to capture whether credit conditions are becoming tighter or looser.


In [66]:
features["credit_spread_change_3m"] = (
    df["credit_spread"].diff(3)
)

features["credit_spread_change_6m"] = (
    df["credit_spread"].diff(6)
)


### Add Forecasting Target


In [67]:
features["target"] = df["recession_6m"]

features.head(15)


,unemployment_rate,fed_funds_rate,yield_spread_10y_2y,yield_spread_10y_3m,consumer_sentiment,credit_spread,unemployment_change_3m,unemployment_change_6m,unemployment_change_12m,initial_claims,...,industrial_production_growth_12m,housing_starts_change_3m,housing_starts_change_12m,consumer_sentiment_change_3m,consumer_sentiment_change_6m,yield_curve_inverted_10y_2y,yield_curve_inverted_10y_3m,credit_spread_change_3m,credit_spread_change_6m,target
1986-01-01,6.7,8.14,1.051429,1.891905,95.6,2.241429,NaN,NaN,NaN,366750.0,...,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,0.0
1986-02-01,7.2,7.86,0.736842,1.413684,95.9,2.408421,NaN,NaN,NaN,369500.0,...,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,0.0
1986-03-01,7.2,7.48,0.564000,1.014500,95.1,2.716500,NaN,NaN,NaN,381200.0,...,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,0.0
1986-04-01,7.1,6.99,0.604091,1.063636,96.2,2.891364,0.4,NaN,NaN,385750.0,...,NaN,-1.977688,NaN,0.6,NaN,0,0,0.649935,NaN,0.0
1986-05-01,7.2,6.85,0.642381,1.374762,94.8,2.576190,0.0,NaN,NaN,386000.0,...,NaN,0.324675,NaN,-1.1,NaN,0,0,0.167769,NaN,0.0
1986-06-01,7.2,6.92,0.614762,1.402857,99.3,2.540000,0.0,NaN,NaN,370500.0,...,NaN,-1.545842,NaN,4.2,NaN,0,0,-0.176500,NaN,0.0
1986-07-01,7.0,6.56,0.636818,1.303636,97.7,2.861364,-0.1,0.3,NaN,362000.0,...,NaN,-7.811692,NaN,1.5,2.1,0,0,-0.030000,0.619935,0.0
1986-08-01,6.9,6.17,0.839524,1.480476,94.9,3.016190,-0.3,-0.3,NaN,407400.0,...,NaN,-2.535059,NaN,0.1,-1.0,0,0,0.440000,0.607769,0.0
1986-09-01,7.0,5.89,1.101429,2.095238,91.9,2.754286,-0.2,-0.2,NaN,395500.0,...,NaN,-8.662696,NaN,-7.4,-3.2,0,0,0.214286,0.037786,0.0
1986-10-01,7.0,5.85,1.147273,2.103182,95.6,2.809545,0.0,-0.1,NaN,383250.0,...,NaN,-5.667789,NaN,-2.1,-0.6,0,0,-0.051818,-0.081818,0.0


In [68]:
features.shape


(487, 24)

## 3. Missing Values

Feature transformations that use previous months naturally create missing values at the beginning of the dataset. The six-month forecasting target also creates missing values at the end because the future recession status is not available within the dataset.

These observations are removed rather than filled with artificial values.


In [69]:
features.isna().sum()


unemployment_rate                    0
fed_funds_rate                       0
yield_spread_10y_2y                  0
yield_spread_10y_3m                  0
consumer_sentiment                   0
credit_spread                        0
unemployment_change_3m               3
unemployment_change_6m               6
unemployment_change_12m             12
initial_claims                       0
initial_claims_change_3m             3
initial_claims_change_6m             6
inflation_yoy                       12
industrial_production_growth_3m      3
industrial_production_growth_12m    12
housing_starts_change_3m             3
housing_starts_change_12m           12
consumer_sentiment_change_3m         3
consumer_sentiment_change_6m         6
yield_curve_inverted_10y_2y          0
yield_curve_inverted_10y_3m          0
credit_spread_change_3m              3
credit_spread_change_6m              6
target                               6
dtype: int64

In [70]:
features_clean = features.dropna().copy()

print("Before:", features.shape)
print("After:", features_clean.shape)


Before: (487, 24)
After: (469, 24)


In [71]:
features_clean.isna().sum().sum()


np.int64(0)

### Target Distribution

After constructing the features and removing incomplete observations, the target distribution is checked again.


In [72]:
features_clean["target"].value_counts()


target
0.0    433
1.0     36
Name: count, dtype: int64

In [73]:
features_clean["target"].value_counts(normalize=True) * 100


target
0.0    92.324094
1.0     7.675906
Name: proportion, dtype: float64

## 4. Avoiding Data Leakage

All predictor variables are constructed using information from the current month or previous months. The target is shifted six months into the future.

This ensures that future recession labels are not directly included in the predictor variables. Maintaining this temporal ordering is essential because using future information would create data leakage and produce unrealistic model performance.

A further limitation is that this project uses currently available historical FRED data rather than real-time data vintages. Economic series may be revised after their original release, and publication delays are not explicitly modeled. This should be considered when interpreting the forecasting results.


In [74]:
features_clean.head()


,unemployment_rate,fed_funds_rate,yield_spread_10y_2y,yield_spread_10y_3m,consumer_sentiment,credit_spread,unemployment_change_3m,unemployment_change_6m,unemployment_change_12m,initial_claims,...,industrial_production_growth_12m,housing_starts_change_3m,housing_starts_change_12m,consumer_sentiment_change_3m,consumer_sentiment_change_6m,yield_curve_inverted_10y_2y,yield_curve_inverted_10y_3m,credit_spread_change_3m,credit_spread_change_6m,target
1987-01-01,6.6,6.43,0.858500,1.506500,90.4,2.642500,-0.4,-0.4,-0.1,345600.0,...,0.610066,5.532421,-10.040568,-5.2,-7.3,0,0,-0.167045,-0.218864,0.0
1987-02-01,6.6,6.10,0.852632,1.499474,90.2,2.402632,-0.3,-0.3,-0.6,348750.0,...,2.656765,9.919901,-3.463203,-1.2,-4.7,0,0,-0.409591,-0.613559,0.0
1987-03-01,6.6,6.13,0.824091,1.482727,90.8,2.361364,0.0,-0.4,-0.6,333250.0,...,3.542961,-5.837425,-7.995736,1.7,-1.1,0,0,-0.498182,-0.392922,0.0
1987-04-01,6.3,6.37,0.998571,2.207143,92.8,2.020476,-0.3,-0.7,-0.8,336250.0,...,4.049323,-9.019166,-16.502845,2.4,-2.8,0,0,-0.622024,-0.789069,0.0
1987-05-01,6.3,6.85,0.851500,2.766500,91.1,1.900000,-0.3,-0.6,-0.9,331000.0,...,4.552761,-8.744395,-12.189860,0.9,-0.3,0,0,-0.502632,-0.912222,0.0


## 5. Save Feature Dataset

The engineered dataset is saved for use in the modeling stage.


In [75]:
features_clean.to_csv(
    "../data/processed/recession_features_6m.csv"
)

print("Feature dataset saved")
print(f"Observations: {len(features_clean)}")
print(f"Features: {features_clean.shape[1] - 1}")
print(
    f"Period: {features_clean.index.min().date()} "
    f"to {features_clean.index.max().date()}"
)


Feature dataset saved
Observations: 469
Features: 23
Period: 1987-01-01 to 2026-02-01
